# Traffic Volume Prediction (Phase 1) - Team Path Protectors

In [ ]:
## import the required packages
from datetime import datetime as dt
import tensorflow as tf

In [ ]:
# define all the constants
GCS_BUCKET = "path-protectors-datalake"
OBJECT_DETECTION_RAW_FILE_PATH = "object-detection/raw"
TARGET_CAMERA_SUBFOLDER = "Stn_HD_1"

In [ ]:
# Define the column names and types
TIMESTAMP = "Timestamp"
VEHICLE_ENTRY = "Vehicle Entry"
VEHICLE_EXIT = "Vehicle Exit"
BICYCLE = "Bicycle"
BUS = "Bus"
CAR = "Cars"
TWO_WHEELER = "Two-Wheeler"
THREE_WHEELER = "Three-Wheeler"
LCV = "LCV"
TRUCK = "Truck"
CAMERA_NAME = "Camera Name"

CSV_COLUMNS = ["", VEHICLE_ENTRY, VEHICLE_EXIT, TIMESTAMP, BICYCLE, BUS, CAR, TWO_WHEELER, THREE_WHEELER, LCV, TRUCK, CAMERA_NAME]
DEFAULTS = [tf.int32, tf.string, tf.string, tf.string, tf.int32, tf.int32, tf.int32,  tf.int32, tf.int32, tf.int32, tf.int32, tf.string]

In [ ]:
# define the target GCS path to your CSV files
gcs_path = f"gs://{GCS_BUCKET}/{OBJECT_DETECTION_RAW_FILE_PATH}/{TARGET_CAMERA_SUBFOLDER}/*.csv"

In [ ]:
gcs_path

In [ ]:
# Process the dataset
def process_features(features):
    # Convert Timestamp to datetime and extract features
    features.pop("")
    timestamp = features.pop(TIMESTAMP)
    # timestamp = dt.strptime(timestamp, "%Y-%m-%d %H:%M:%S") 
    features['Month'] = timestamp.month
    features['Date'] = timestamp.day
    features['Hour'] = timestamp.hour
    features['Minute'] = timestamp.minute
    

    # One-hot encode categorical columns
    vehicle_entry = tf.one_hot(tf.strings.to_hash_bucket_fast(features.pop(VEHICLE_ENTRY), num_buckets=10), depth=10)
    vehicle_exit = tf.one_hot(tf.strings.to_hash_bucket_fast(features.pop(VEHICLE_EXIT), num_buckets=10), depth=10)
    camera_name = tf.one_hot(tf.strings.to_hash_bucket_fast(features.pop(CAMERA_NAME), num_buckets=10), depth=10)
    
    day_str = timestamp.strftime("%A")
    day = tf.one_hot(tf.strings.to_hash_bucket_fast(day_str, num_buckets=10), depth=10)
    print(features.values())
    features = tf.concat([vehicle_entry, vehicle_exit, camera_name, tf.stack(list(features.values()), axis=1)], axis=1)
    
    return features

In [ ]:
# Create the dataset
dataset = tf.data.experimental.make_csv_dataset(
    file_pattern=gcs_path,
    batch_size=8,
    column_names=CSV_COLUMNS,
    column_defaults=DEFAULTS,
    label_name=None,
    num_epochs=1,
    header=True
)

In [ ]:
dataset

In [ ]:
# process and map the tensors via process_features function
dataset = dataset.map(process_features)

In [ ]:
# Batch and shuffle the dataset
dataset = dataset.shuffle(buffer_size=1000).batch(8).prefetch(tf.data.experimental.AUTOTUNE)

In [ ]:
# Example of iterating through the dataset
for batch in dataset.take(1):
    print(batch)

In [ ]:
print(dataset.__len__)

In [ ]:
for batch in dataset.take(1):
    print(len(batch))
    for key, value in batch.items():
        print(f"{key:20s}: {value}")
    print()